# Evaluation: LLMs and BERT Models on 16 Hate Speech Test Sets

This notebook evaluates fine-tuned LLM models (Llama3.2-1B and Qwen2.5-14B) and BERT-based models
on 16 benchmark hate speech test sets across English, German, Spanish, and Vietnamese.

**Evaluation tracks:**
- **LLM track**: Load each fine-tuned model variant (Human, LGB, Mean, Vote, Human-LGB), run inference
  using token probability extraction, and compute macro-F1 per dataset.
- **BERT track**: Fine-tune or load a pre-fine-tuned BERT model, run inference, and generate a per-dataset report.

Pre-computed probability outputs are saved as `.pkl` files in this folder for reproducibility.

## 1. Imports and Configuration

Import all required libraries, set the random seed for reproducibility, and configure the device
(CUDA if available). The `FastLanguageModel` class from Unsloth is used for memory-efficient
4-bit quantized inference.

In [ ]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset

from transformers import (
    set_seed,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments, 
    Trainer,
)
from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import torch
from datasets import load_dataset
def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)
import torch.nn.functional as F


device = 'cuda' if torch.cuda.is_available() else 'cpu'


# 1 LLMs

Selecting batch size

In [12]:
repo = "danghaidang-passau/HateOWS-dataset-LREC2026"
ds_16 = load_dataset(repo, "16-val-train")       
ds_train = ds_16["train"]
ds_val = ds_16["test"]


Generating test split: 100%|██████████| 31365/31365 [00:00<00:00, 1809326.84 examples/s]


In [13]:
df_train =  ds_16["train"].to_pandas()
df_val =  ds_16["test"].to_pandas()

In [129]:
df_val.ds.value_counts()

ds
HateXplain      3846
GermEval18      3532
AHSD            3000
ViHSD           2672
Sexism          2632
GermEval19      2507
Gahd            2198
GermEval21      2085
Chileno         1928
HateEval-spa    1286
Haternet        1205
US_election     1117
HateEval-eng    1000
Covid            971
AbusEval         860
HASOC            526
Name: count, dtype: int64

In [130]:
print(df_val.groupby(['ds', 'label_id']).size().unstack(fill_value=0))

label_id         1     2
ds                      
AHSD          2494   506
AbusEval       178   682
Chileno        110  1818
Covid          190   781
Gahd           939  1259
GermEval18    1202  2330
GermEval19     840  1667
GermEval21     769  1316
HASOC          134   392
HateEval-eng   427   573
HateEval-spa   556   730
HateXplain    2283  1563
Haternet       305   900
Sexism         350  2282
US_election    140   977
ViHSD          482  2190


# 2 Bert

In [131]:

def classify_texts_Bert(texts=None, batch_size=128, model=None, tokenizer=None, return_probs=False):
    predictions = []
    probs_all = []
    model = model.to("cuda")
    model.eval()
    print()

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        encoded = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True, max_length=512)
        encoded = {k: v.to("cuda") for k, v in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1) 
            batch_preds = torch.argmax(probs, dim=1).tolist()
            batch_probs = probs.tolist()
            # batch_preds = torch.argmax(logits, dim=1).tolist()
            predictions.extend(batch_preds)
            probs_all.extend(batch_probs)
    probs_all = np.vstack(probs_all)
    if return_probs:
        return probs_all
    return predictions

### BERT Fine-Tuning Function

**`finetuneBert`**: Fine-tunes a BERT-based model for binary hate speech classification using Hugging Face
`Trainer`. The function:
1. Shuffles and tokenizes the input dataframe
2. Filters out sequences longer than 512 tokens using a dynamic data collator with padding
3. Trains with a linear learning rate schedule and saves the best checkpoint by F1

After fine-tuning, use `classify_texts_Bert` to run inference and `make_report` to compute per-dataset results.

In [134]:
def finetuneBert(
        df=None,
        base=None,
        tokenizer=None,
        save_path=None,
):
    
    df = df.sample(frac=1, random_state=def_seed).reset_index(drop=True)

    dataset = Dataset.from_pandas(df)

    def tokenize_and_format(batch):
        encodings = tokenizer(batch["text"] )
        encodings["len"] = [len(ids) for ids in encodings["input_ids"]]
        encodings["labels"] = batch["labels"] 
        return encodings


    tokenized_dataset = dataset.map(
        tokenize_and_format,
        batched = True,
        batch_size = 512,
        num_proc=10
    )

    filtered_dataset = tokenized_dataset.filter(
        lambda example: example['len'] < 512
    )

    training_args = TrainingArguments(
        output_dir=save_path,
        overwrite_output_dir=True,
        # num_train_epochs=train_conf["num_train_epochs"],
        num_train_epochs=3,
        # max_steps=2000,
        save_strategy="steps",        
        save_steps=2000,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=1,
        learning_rate=1e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.2,
        max_grad_norm=0.2,
        logging_steps=200,
        fp16=False,
        bf16=True,
        gradient_accumulation_steps=2,
        gradient_checkpointing=True,
        report_to="none",              
        eval_steps=5000,
        max_steps=-1,                  
        log_level="debug",
        dataloader_num_workers=2,
    )


    trainer = Trainer(
        model=base,
        args=training_args,
        train_dataset=filtered_dataset,
        tokenizer=tokenizer,
    )

    trainer.train()
    return trainer
    

Labeling for Ows4L Bert, has been trained with all 16 dataset

In [ ]:
df_train.ds.value_counts()

ds
AHSD            21783
HateXplain      15299
AbusEval        13240
Sexism          10904
GermEval19       9698
HateEval-eng     9000
Gahd             8797
ViHSD            8061
Chileno          7572
HateEval-spa     5309
GermEval18       5009
Haternet         4794
HASOC            2373
GermEval21       2071
US_election      1283
Covid            1282
Name: count, dtype: int64

In [ ]:
df_train.iloc[0]

text        @dualipuhs Leave my ship alone you whore https...
language                                                  eng
ds                                               HateEval-eng
label_id                                                    1
Name: 0, dtype: object

In [143]:
Ows4L = "danghaidang-passau/Ows4L_16"
OwsSpa = "danghaidang-passau/OwsSpa"
OwsDeu = "danghaidang-passau/OwsDeu"
OwsEng = "danghaidang-passau/OwsEng"

HateBert = "GroNLP/hateBERT"
Bert = "google-bert/bert-base-uncased"



# Fine-tuning setup — 5 base models × 6 validation groupings

We fine-tune five base models across six dataset groupings (used as validation sets) to evaluate robustness and cross-dataset generalization.

**Base models (pretrained checkpoints / hubs):**
- `Ows4L` — danghaidang-passau/Ows4L_16
- `OwsSpa` — danghaidang-passau/OwsSpa
- `OwsDeu` — danghaidang-passau/OwsDeu
- `OwsEng` — danghaidang-passau/OwsEng
- `HateBert` — GroNLP/hateBERT
(optional baseline: `Bert` — google-bert/bert-base-uncased)

**Validation groupings (use as validation/hold-out sets during fine-tuning):**
1. English group set (Eng)
2. Spanish group set (Spa)
3. German group set (Deu)
4. 7-set mixed corpus (7set)
5. 7-set + synthetic data (7set+synthetic)
6. All 16 human validation datasets (16-val)

**Procedure (per model × per grouping):**
- Train on the chosen training splits and validate on the grouping listed above.
- Use a distinct output directory per run, e.g. `outputs/{model_name}/{group_name}`.
- Log metrics (macro-F1, precision, recall, accuracy) per validation dataset and save the best checkpoint by F1.

**Example: load a base model and tokenizer before fine-tuning**
```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model_name = "danghaidang-passau/Ows4L_16"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_name)
```

Keep configuration (learning rate, batch size, number of epochs, scheduler, checkpointing) consistent across runs where possible, and vary one experimental factor at a time when testing robustness (for example: validation grouping, synthetic data inclusion, or augmentation strategy).

In [ ]:
def training_model_with_df_train(df_train, model_name, save_path, df_val):
    # Load the pre-trained model and tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    trainer = finetuneBert(
        df_train,
        model,
        tokenizer,
       save_path
    )
    
    # Load the fine-tuned model for evaluation
    model = AutoModelForSequenceClassification.from_pretrained(save_path, num_labels=2)
    output_pred = classify_texts_Bert(
        texts=df_val['text'].fillna('').tolist(),
        model=model,
        tokenizer=tokenizer
    )
    return output_pred

In [137]:
df_eng = df_train[df_train['language'] == 'eng']
df_eng.ds.value_counts()

ds
AHSD            21783
HateXplain      15299
AbusEval        13240
Sexism          10904
HateEval-eng     9000
US_election      1283
Covid            1282
Name: count, dtype: int64

In [146]:
df_deu = df_train[df_train['language'] == 'deu']
df_deu.ds.value_counts()

ds
GermEval19    9698
Gahd          8797
GermEval18    5009
HASOC         2373
GermEval21    2071
Name: count, dtype: int64

In [147]:
df_spa = df_train[df_train['language'] == 'spa']
df_spa.ds.value_counts()

ds
Chileno         7572
HateEval-spa    5309
Haternet        4794
Name: count, dtype: int64

In [166]:
selected = ["HateXplain", "Sexism", "US_election", "Covid", "GermEval21", "GermEval19", "ViHSD"]
df_7_subset = df_train[df_train["ds"].isin(selected)].copy()

In [169]:
ds_46k_sythentic = load_dataset(repo, "46k_qwen")       
df_46k_sythentic = ds_46k_sythentic["train"]

README.md: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
model_probs_dict = {}
train_set_key = ['spa', 'deu', 'eng', '7set', '16set', '7fake']
for key in train_set_key:
    model_probs_dict[key] = {}

# Finetune only English test set with OWS_eng, Bert and HateBert

In [ ]:
model_probs_dict["eng"]["OWS_eng"] = training_model_with_df_train(df_eng, OwsEng, "eng_OWS_eng", df_val)
model_probs_dict["eng"]["Bert"] = training_model_with_df_train(df_eng, Bert, "eng_Bert", df_val)
model_probs_dict["eng"]["HateBert"] = training_model_with_df_train(df_eng, HateBert, "eng_HateBert", df_val)

# Finetune only German test set with OWS_deu, Bert and HateBert

In [ ]:
model_probs_dict["deu"]["OWS_deu"] = training_model_with_df_train(df_eng, OwsDeu, "deu_OWS_deu", df_val)
model_probs_dict["deu"]["Bert"] = training_model_with_df_train(df_eng, Bert, "deu_Bert", df_val)
model_probs_dict["deu"]["HateBert"] = training_model_with_df_train(df_eng, HateBert, "deu_HateBert", df_val)

# Finetune only Spanish test set with OWS_Spa, Bert and HateBert

In [ ]:
model_probs_dict["spa"]["OWS_spa"] = training_model_with_df_train(df_eng, OwsSpa, "spa_OWS_spa", df_val)
model_probs_dict["spa"]["Bert"] = training_model_with_df_train(df_eng, Bert, "spa_Bert", df_val)
model_probs_dict["spa"]["HateBert"] = training_model_with_df_train(df_eng, HateBert, "spa_HateBert", df_val)

# Finetune only 7 test set with OWS_4L, Bert and HateBert

In [ ]:
model_probs_dict["7set"]["OWS_4L"] = training_model_with_df_train(df_7_subset, Ows4L, "7set_OWS_4L", df_val)
model_probs_dict["7set"]["Bert"] = training_model_with_df_train(df_7_subset, Bert, "7set_Bert", df_val)
model_probs_dict["7set"]["HateBert"] = training_model_with_df_train(df_7_subset, HateBert, "7set_HateBert", df_val)

# Finetune only 7 set + 46K sythentic annotated with Qwen2.4-14B model with OWS_4L, Bert and HateBert

In [ ]:
df_7_fake = pd.concat([df_7_subset, df_46k_sythentic], ignore_index=True)
model_probs_dict["7fake"]["OWS_4L"] = training_model_with_df_train(df_7_fake, Ows4L, "7fake_OWS_4L", df_val)
model_probs_dict["7fake"]["Bert"] = training_model_with_df_train(df_7_fake, Bert, "7fake_Bert", df_val)
model_probs_dict["7fake"]["HateBert"] = training_model_with_df_train(df_7_fake, HateBert, "7fake_HateBert", df_val)

# Finetune all 16 set with OWS_4L, Bert and HateBert

In [ ]:
model_probs_dict["16set"]["OWS_4L"] = training_model_with_df_train(df_train, Ows4L, "16set_OWS_4L", df_val)
model_probs_dict["16set"]["Bert"] = training_model_with_df_train(df_train, Bert, "16set_Bert", df_val)
model_probs_dict["16set"]["HateBert"] = training_model_with_df_train(df_train, HateBert, "16set_HateBert", df_val)

In [16]:
with open("Bert_probs.pkl", "rb") as f:
    model_probs_dict = pickle.load(f)

In [17]:
for k, v in model_probs_dict.items():
    model_probs_dict[k]["HateBert"] = v["Hate"]
    del model_probs_dict[k]["Hate"]

In [18]:
df_eval = df_val
report = {}
for ds in df_eval['ds'].unique():
    report[ds] = {}
report["Mean"] = {}

y_true = np.array(df_eval['label_id'] == 1, dtype=int)
df_eval['y_ture'] = y_true

for test_type, model_list in model_probs_dict.items():
        for model_id, y_pred in model_list.items():
            df_eval['y_pred'] = y_pred
            report["Mean"][test_type + "_" + model_id] = {
            "f1": round(f1_score(y_true,y_pred, average='macro')* 100, 1)
                }
            
            for ds in df_eval['ds'].unique():
                tmp_df = df_eval.loc[df_eval['ds'] == ds]
                y_pred = tmp_df['y_pred']

                f1 = round(f1_score(tmp_df['y_ture'], y_pred, average='macro')* 100, 1) 
                report[ds][test_type + "_" + model_id] = {
                                "f1": f1
                                }

In [19]:
report_df = pd.DataFrame(report).T
report_df

,spa_OWS_spa,spa_Bert,spa_HateBert,deu_OWS_deu,deu_Bert,deu_HateBert,eng_Bert,eng_OWS_eng,eng_HateBert,7set_Bert,7set_OWS_4L,7set_HateBert,16set_Bert,16set_OWS_4L,16set_HateBert,7fake_Bert,7fake_OWS_4L,7fake_HateBert
HateXplain,{'f1': 28.9},{'f1': 30.3},{'f1': 54.7},{'f1': 59.7},{'f1': 52.6},{'f1': 55.5},{'f1': 77.7},{'f1': 77.9},{'f1': 77.5},{'f1': 77.7},{'f1': 77.6},{'f1': 77.6},{'f1': 78.0},{'f1': 77.5},{'f1': 77.1},{'f1': 78.9},{'f1': 78.4},{'f1': 77.8}
GermEval18,{'f1': 39.7},{'f1': 40.1},{'f1': 39.8},{'f1': 81.4},{'f1': 76.4},{'f1': 72.8},{'f1': 42.0},{'f1': 41.0},{'f1': 42.4},{'f1': 80.5},{'f1': 83.1},{'f1': 75.5},{'f1': 80.5},{'f1': 82.6},{'f1': 75.7},{'f1': 76.3},{'f1': 78.6},{'f1': 74.6}
Haternet,{'f1': 68.1},{'f1': 67.3},{'f1': 64.9},{'f1': 56.8},{'f1': 45.7},{'f1': 48.7},{'f1': 43.0},{'f1': 44.4},{'f1': 45.1},{'f1': 45.2},{'f1': 48.0},{'f1': 46.7},{'f1': 69.3},{'f1': 71.9},{'f1': 67.7},{'f1': 47.2},{'f1': 49.1},{'f1': 45.2}
AHSD,{'f1': 14.4},{'f1': 16.1},{'f1': 39.5},{'f1': 45.3},{'f1': 28.8},{'f1': 48.5},{'f1': 91.7},{'f1': 91.7},{'f1': 90.6},{'f1': 49.4},{'f1': 54.0},{'f1': 50.6},{'f1': 91.4},{'f1': 91.9},{'f1': 91.0},{'f1': 78.4},{'f1': 79.4},{'f1': 79.1}
AbusEval,{'f1': 44.2},{'f1': 44.2},{'f1': 53.5},{'f1': 51.4},{'f1': 48.6},{'f1': 58.6},{'f1': 67.0},{'f1': 70.7},{'f1': 67.1},{'f1': 52.1},{'f1': 52.5},{'f1': 55.1},{'f1': 68.6},{'f1': 72.1},{'f1': 67.1},{'f1': 51.7},{'f1': 55.9},{'f1': 56.7}
GermEval19,{'f1': 39.9},{'f1': 40.0},{'f1': 39.9},{'f1': 78.8},{'f1': 74.8},{'f1': 70.8},{'f1': 41.7},{'f1': 41.3},{'f1': 41.7},{'f1': 73.4},{'f1': 74.6},{'f1': 69.8},{'f1': 77.5},{'f1': 79.1},{'f1': 73.4},{'f1': 72.0},{'f1': 73.5},{'f1': 71.0}
ViHSD,{'f1': 45.0},{'f1': 45.0},{'f1': 45.7},{'f1': 55.5},{'f1': 49.1},{'f1': 50.9},{'f1': 45.2},{'f1': 45.0},{'f1': 45.6},{'f1': 72.2},{'f1': 74.0},{'f1': 67.4},{'f1': 72.3},{'f1': 73.8},{'f1': 67.9},{'f1': 71.0},{'f1': 72.8},{'f1': 64.7}
HateEval-eng,{'f1': 36.4},{'f1': 37.4},{'f1': 54.0},{'f1': 58.8},{'f1': 47.2},{'f1': 56.4},{'f1': 77.3},{'f1': 75.9},{'f1': 75.6},{'f1': 58.7},{'f1': 60.0},{'f1': 60.5},{'f1': 77.0},{'f1': 76.2},{'f1': 76.3},{'f1': 60.2},{'f1': 61.4},{'f1': 62.2}
HateEval-spa,{'f1': 76.6},{'f1': 75.9},{'f1': 73.5},{'f1': 46.9},{'f1': 38.3},{'f1': 38.4},{'f1': 36.2},{'f1': 36.3},{'f1': 36.7},{'f1': 36.9},{'f1': 38.6},{'f1': 36.7},{'f1': 78.6},{'f1': 79.0},{'f1': 76.7},{'f1': 38.5},{'f1': 38.2},{'f1': 36.5}
Gahd,{'f1': 36.9},{'f1': 37.7},{'f1': 36.5},{'f1': 73.2},{'f1': 71.0},{'f1': 70.0},{'f1': 39.2},{'f1': 38.9},{'f1': 40.8},{'f1': 53.2},{'f1': 55.5},{'f1': 53.9},{'f1': 72.9},{'f1': 73.2},{'f1': 71.2},{'f1': 45.7},{'f1': 45.9},{'f1': 46.5}
